In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

In [2]:
# Step 1 — Load your cleaned Titanic datase
df = pd.read_csv("../ML/titanic_cleaned_training_data_FE.csv")

X = df.drop("Survived", axis=1)
y = df["Survived"]

In [3]:
# Step 2 — Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [4]:
# Step 4 — Define the models you want to compare
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=300),
    "Gradient Boosting": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}

In [5]:
# 5 Preprpcess/ ColumnTransformer

# Identify your categorical and numeric columns
categorical_cols = X.select_dtypes(include=["object"]).columns
numeric_cols = X.select_dtypes(exclude=["object"]).columns

# Build a ColumnTransformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

C:\Users\gkgab\AppData\Local\Temp\ipykernel_42296\2132796921.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns


In [6]:
# # Step 6 — Build a pipeline (scaling included)

# pipelines = {}

# for name, model in models.items():
#     if name in ["Logistic Regression", "KNN"]:
#         pipelines[name] = Pipeline([("scaler", StandardScaler()), ("model", model)])
#     else:
#         pipelines[name] = Pipeline([("model", model)])

# Wrap each model inside a pipeline that starts with the preprocessor
pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

In [7]:
# Step 7 — Train and evaluate each model

results = []

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    results.append(
        {
            "Model": name,
            "Accuracy": accuracy_score(y_test, preds),
            "Precision": precision_score(y_test, preds),
            "Recall": recall_score(y_test, preds),
            "F1 Score": f1_score(y_test, preds),
        }
    )

results_df = pd.DataFrame(results)
results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.843575,0.825397,0.753623,0.787879
1,Random Forest,0.832402,0.819672,0.724638,0.769231
2,Gradient Boosting,0.826816,0.839286,0.681159,0.752000
3,KNN,0.793296,0.735294,0.724638,0.729927
